# Bone Analysing
Notebook to analyze the bone, liver and organ metastasis

First let's get a table of the spacings for each of the cases so we can easily calculate volumes later on

# Collate the bone metastasis information

In [3]:
SAVE_DIR = "//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/{dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold_name}/bone_analysis/"

# Helper function to get the mets and metrics files that are generated by bone_analysis/lesion_metrics_new.sh
# This can be run instead of the normal nnUNet.evaluation file 

def get_mets_files(dataset, fold_name):
    mets_dir = SAVE_DIR.format(dataset=dataset, fold_name=fold_name)
    if not os.path.exists(mets_dir):
        print(f"Directory {mets_dir} does not exist.")
        return []
    
    mets_files = [os.path.join(mets_dir, f) for f in os.listdir(mets_dir) if f.endswith('mets.csv')]
    metrics_files = [f.replace('mets.csv', 'mets_metrics.csv') for f in mets_files]

    print(f"Found {len(mets_files)} METS files in {mets_dir}.")
    return mets_files, metrics_files

In [4]:
import pandas as pd
# helper functions to preprocess the loading of the files

def _preprocess_mets_file(mets_file):
    try:
        mets_df = pd.read_csv(mets_file)
        case_name = os.path.basename(mets_file).replace('_mets', '')
        # for the columns with Volume, divide by 1000 to convert from mm^3 to cm^3
        mets_df['file'] = case_name
        for col in mets_df.columns:
            if 'Volume' in col or 'mm3' in col:
                mets_df[col] = mets_df[col] / 1000.0
        return mets_df

    except Exception as e:
        print(f"Error reading {mets_file}: {e}")
        return None

def _preprocess_metrics_file(metrics_file): 
    try: 
        metrics_df = pd.read_csv(metrics_file)
    
        for col in metrics_df.columns:
            if 'vol' in col or 'mm3' in col:
                metrics_df[col] = metrics_df[col] / 1000.0
        
        metrics_df['file'] = metrics_df['file'].str.replace('_0000', '')
    except Exception as e:
        print(f"Error reading {metrics_file}: {e}")
        return None

    return metrics_df


def hr_organ_analysis(mets_df):
    if mets_df is None:
        return None, None
    lung_df = mets_df.loc[mets_df['Description'].str.contains('lung', case=False, na=False)     ]
    liver_df = mets_df.loc[mets_df['Description'] == 'liver']
    return lung_df, liver_df

def hr_bone_analysis(mets_df):
    if mets_df is None:
        return None, None
    priority_bones  = [25,26,27,31,32,43,44,69,70,71,72,73,74,75,76,77,78]
    bones_df = mets_df.loc[mets_df['Type'].str.contains('bone', case=False, na=False)]
    hr_bones_df = bones_df.loc[bones_df['Label'].isin(priority_bones)]
    return bones_df, hr_bones_df


In [106]:
def get_global_metrics(mets_files: list, metrics_files: list): 
    """
    Returns global metrics (dice, fp vol, fn vol)
    mets metrics (individual stats per organ per case)
    liver metrics (individual stats filtered for liver)
    lung metrics (individual stats filtered for lung)
    hr bones metrics (individual stats filtered for high risk bones)
    bones metrics (individual stats filtered for all bones)

    """
    global_mets = pd.DataFrame()
    global_liver = pd.DataFrame()
    global_lung = pd.DataFrame()
    global_bones = pd.DataFrame()
    global_hr_bones = pd.DataFrame()
    global_metrics = pd.DataFrame()

    for idx, _ in enumerate(mets_files):
        temp_mets_df = _preprocess_mets_file(mets_files[idx])
        temp_metrics_df = _preprocess_metrics_file(metrics_files[idx])
        
        lung_df, liver_df = hr_organ_analysis(temp_mets_df)
        bones_df, hr_bones_df = hr_bone_analysis(temp_mets_df)
        
        global_metrics = pd.concat([global_metrics, temp_metrics_df], ignore_index=True)
        global_mets = pd.concat([global_mets, temp_mets_df], ignore_index=True)
        global_liver = pd.concat([global_liver, liver_df], ignore_index=True)
        global_lung = pd.concat([global_lung, lung_df], ignore_index=True)
        global_bones = pd.concat([global_bones, bones_df], ignore_index=True)
        global_hr_bones = pd.concat([global_hr_bones, hr_bones_df], ignore_index=True)
    
    return global_metrics, global_mets, global_liver, global_lung, global_bones, global_hr_bones

class Metrics:    
    def __init__(self, dataset, fold, global_metrics, global_mets, global_liver, global_lung, global_bones, global_hr_bones):
        self.dataset = dataset
        self.fold = fold
        
        self.global_metrics = global_metrics
        self.global_mets = global_mets
        self.global_liver = global_liver
        self.global_lung = global_lung
        self.global_bones = global_bones
        self.global_hr_bones = global_hr_bones
    
        self.liver_cases = pd.read_csv('unique_liver_cases.csv').replace('.csv', '')
        self.lung_cases = pd.read_csv('unique_lung_cases.csv').replace('.csv', '')
        self.bone_cases = pd.read_csv('unique_bone_cases.csv').replace('.csv', '')
        self.hr_bone_cases = pd.read_csv('unique_hr_bone_cases.csv').replace('.csv', '')
        
        self.global_liver = self.global_liver[self.global_liver['file'].isin(self.liver_cases['file'])]
        self.global_lung = self.global_lung[self.global_lung['file'].isin(self.lung_cases['file'])]
        self.global_bones = self.global_bones[self.global_bones['file'].isin(self.bone_cases['file'])]
        self.global_hr_bones = self.global_hr_bones[self.global_hr_bones['file'].isin(self.hr_bone_cases['file'])]
        
        self.adjust_liver(overlap_threshold=0.1)
        self.adjust_lung(overlap_threshold=0.1)
        self.adjust_bones(overlap_threshold=0.1)
        self.adjust_hr_bones(overlap_threshold=0.1)
        
    def get_fold_level_metrics(self): 
        return{
            "Dice": self.global_metrics.loc[self.global_metrics['gt_lesion_count'] > 0, 'dice_sc'].mean(),
            "FP_vol_avg (cm3)": self.global_metrics['false_pos_vol'].mean(),
            "FN_vol_avg (cm3)": self.global_metrics['false_neg_vol'].mean(),
            "FP_vol_total (cm3)": self.global_metrics['false_pos_vol'].sum(),
            "FN_vol_total (cm3)": self.global_metrics['false_neg_vol'].sum(), 
            "Total predicted lesions": self.global_metrics['pred_lesion_count'].sum()
        }
    
    def get_mets_level_metrics(self, mets_df):
        return {
            "Dice": mets_df['Dice_Score'].mean(),
            "FP_vol_avg (cm3)": mets_df['FP_vol_mm3'].mean(),
            "FN_vol_avg (cm3)": mets_df['FN_vol_mm3'].mean(),
            "FP_vol_total (cm3)": mets_df['FP_vol_mm3'].sum(),
            "FN_vol_total (cm3)": mets_df['FN_vol_mm3'].sum()
            }
    
    def adjust_liver(self, overlap_threshold=0.1):
        self.global_liver = self.global_mets[
        (self.global_mets['Description'] == 'liver') &
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
    
    def adjust_lung(self, overlap_threshold=0.1):
        self.global_lung = self.global_mets[
        (self.global_mets['Description'].str.contains('lung', case=False, na=False)) &
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
    
    def adjust_bones(self, overlap_threshold=0.1):
        self.global_bones = self.global_mets[
        (self.global_mets['Type'].str.contains('bone', case=False, na=False)) &
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
        
    def adjust_hr_bones(self, overlap_threshold=0.1):
        self.global_hr_bones = self.global_mets[
        (self.global_mets['Type'].str.contains('bone', case=False, na=False)) &
        (self.global_mets['Label'].isin([25,26,27,31,32,43,44,69,70,71,72,73,74,75,76,77,78])) & # CHECK THIS 
        ((self.global_mets['Overlap_Volume_ground'] > overlap_threshold) | (self.global_mets['Overlap_Volume_pred'] > overlap_threshold))
    ]
        
            
    

In [107]:
DATASET = "Dataset903_AutoPet"
FOLD_NAME = "fold_0"

mets_files, metric_files = get_mets_files(DATASET, FOLD_NAME)

dat = Metrics(DATASET, FOLD_NAME, *get_global_metrics(mets_files[:5], metric_files[:5]))


Found 321 METS files in //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset903_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/.


In [108]:
dat.global_mets

,Label,Description,Count,Type,Priority,Overlap_Count_ground,Overlap_Volume_ground,Overlap_Percentage_ground,High Risk_ground,Overlap_Count_pred,Overlap_Volume_pred,Overlap_Percentage_pred,High Risk_pred,Intersection_mm3,Dice_Score,FP_Volume_mm3,FN_Volume_mm3,file
0,1,spleen,16633,Organ,True,15,0.186615,0.09,True,245,3.048050,1.47,True,0.124410,0.076923,0.0,0.000000,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...
1,2,kidney_right,10428,Organ,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...
2,3,kidney_left,11913,Organ,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...
3,4,gallbladder,726,Organ,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...
4,5,liver,107843,Organ,True,1636,20.353510,1.52,True,1364,16.969552,1.26,True,15.961829,0.855333,0.0,0.659374,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410,113,rib_right_10,2023,Bone,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_d65cf203be_04-28-2007-NA-PET-CT Ganzkoerpe...
411,114,rib_right_11,1245,Bone,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_d65cf203be_04-28-2007-NA-PET-CT Ganzkoerpe...
412,115,rib_right_12,871,Bone,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_d65cf203be_04-28-2007-NA-PET-CT Ganzkoerpe...
413,116,sternum,6954,Bone,True,0,0.000000,0.00,False,0,0.000000,0.00,False,0.000000,0.000000,0.0,0.000000,fdg_d65cf203be_04-28-2007-NA-PET-CT Ganzkoerpe...


In [104]:
dat.get_mets_level_metrics(dat.global_liver)

KeyError: 'dice_sc'

In [90]:
dat.get_fold_level_metrics()

{'Dice': np.float64(0.756312350538962),
 'FP_vol_avg (cm3)': np.float64(0.2214501658827553),
 'FN_vol_avg (cm3)': np.float64(5.765218961845433),
 'FP_vol_total (cm3)': np.float64(1.1072508294137764),
 'FN_vol_total (cm3)': np.float64(28.826094809227165),
 'Total predicted lesions': np.int64(41)}

In [91]:
dat.get_mets_level_metrics(dat.global_liver)

KeyError: 'gt_lesion_count'

In [71]:
thingy

,Label,Description,Count,Type,Priority,Overlap_Count_ground,Overlap_Volume_ground,Overlap_Percentage_ground,High Risk_ground,Overlap_Count_pred,Overlap_Volume_pred,Overlap_Percentage_pred,High Risk_pred,Intersection_mm3,Dice_Score,FP_Volume_mm3,FN_Volume_mm3,file
4,5,liver,107843,Organ,True,1636,20.353510,1.52,True,1364,16.969552,1.26,True,15.961829,0.855333,0.000000,0.659374,fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerpe...
1083,5,liver,95139,Organ,True,75,0.933077,0.08,True,0,0.000000,0.00,False,0.000000,0.000000,0.000000,0.933077,fdg_fe705ea1cc_12-29-2002-NA-Unspecified CT AB...
1166,5,liver,139327,Organ,True,34,0.422995,0.02,True,27,0.335908,0.02,True,0.149292,0.393443,0.000000,0.273702,fdg_e69152b6b4_01-26-2004-NA-PET-CT Ganzkoerpe...
1664,5,liver,32685,Organ,True,0,0.000000,0.00,False,1,0.033176,0.00,False,0.000000,0.000000,0.033176,0.000000,psma_0179419e313f7d8c_2019-06-10.csv
1996,5,liver,174940,Organ,True,361,4.491208,0.21,True,165,2.052768,0.09,True,1.853712,0.566540,0.000000,0.000000,fdg_2b60c8135a_12-09-2005-NA-PET-CT Ganzkoerpe...
2494,5,liver,41615,Organ,True,7,0.232231,0.02,True,0,0.000000,0.00,False,0.000000,0.000000,0.000000,0.232231,psma_120974a1c4058f4a_2018-06-11.csv
3158,5,liver,48181,Organ,True,5,0.122246,0.01,True,0,0.000000,0.00,False,0.000000,0.000000,0.000000,0.122246,psma_c81bde6455d07a32_2020-06-22.csv
3490,5,liver,21550,Organ,True,538,44.621574,2.50,True,88,7.298696,0.41,True,6.386359,0.246006,0.000000,25.545437,psma_505ebb983b71c700_2015-05-29.csv
3739,5,liver,30541,Organ,True,8888,737.168313,29.10,True,567,47.026826,1.86,True,44.040996,0.112322,0.331759,34.834686,psma_505ebb983b71c700_2015-11-07.csv
3988,5,liver,92562,Organ,True,0,0.000000,0.00,False,15,0.186615,0.02,False,0.000000,0.000000,0.186615,0.000000,fdg_11e258cc1f_03-06-2003-NA-PET-CT Ganzkoerpe...


In [62]:
DATASET = "Dataset902_AutoPet"
FOLD_NAME = "fold_0"

mets_files, metric_files = get_mets_files(DATASET, FOLD_NAME)

dat2 = Metrics(DATASET, FOLD_NAME, *get_global_metrics(mets_files, metric_files))
dat2.get_metrics()

Found 321 METS files in //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset902_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/.
Error reading //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset902_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/fdg_0b57b247b6_05-02-2002-NA-PET-CT Ganzkoerper  primaer mit KM-42966_mets.csv: No columns to parse from file
Error reading //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset902_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/fdg_0b57b247b6_05-02-2002-NA-PET-CT Ganzkoerper  primaer mit KM-42966_mets_metrics.csv: [Errno 2] No such file or directory: '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset902_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/fdg_0b57b247b6_05-02-2002-

KeyboardInterrupt: 

114

In [ ]:
# get the cases which actually have these things 

# unique_liver_cases = dat_902.global_liver.loc[dat_902.global_liver['Overlap_Volume_ground'] > 1]
# unique_lung_cases = dat_902.global_lung.loc[dat_902.global_lung['Overlap_Volume_ground'] > 1]
# unique_bone_cases = dat_902.global_bones.loc[dat_902.global_bones['Overlap_Volume_ground'] > 1]
# unique_hr_bone_cases = dat_902.global_hr_bones.loc[dat_902.global_hr_bones['Overlap_Volume_ground'] > 1]

# unique_liver_cases[['Label', 'Description', 'Count', 'Type', 'Priority', 'Overlap_Count_ground', 'Overlap_Volume_ground', 'file']].to_csv('unique_liver_cases.csv', index=False)
# unique_lung_cases[['Label', 'Description', 'Count', 'Type', 'Priority', 'Overlap_Count_ground', 'Overlap_Volume_ground', 'file']].to_csv('unique_lung_cases.csv', index=False)
# unique_bone_cases[['Label', 'Description', 'Count', 'Type', 'Priority', 'Overlap_Count_ground', 'Overlap_Volume_ground', 'file']].to_csv('unique_bone_cases.csv', index=False)
# unique_hr_bone_cases[['Label', 'Description', 'Count', 'Type', 'Priority', 'Overlap_Count_ground', 'Overlap_Volume_ground', 'file']].to_csv('unique_hr_bone_cases.csv', index=False)


In [ ]:
unique_bone_cases['file'].values[0]

'fdg_dc6174cb5d_03-29-2003-NA-PET-CT Ganzkoerper  primaer mit KM-09363.csv'

# Collating


In [ ]:
def load_all_datasets(dataset_folds: dict):
    """
    dataset_folds: dict like {
        "Dataset902_AutoPet": ["fold_0", "fold_1"],
        "Dataset903_AutoPet": ["fold_0"],
    }
    Returns a dict of { "Dataset__fold": Metrics }
    """
    all_metrics = {}
    for dataset, folds in dataset_folds.items():
        for fold in folds:
            mets_files, metric_files = get_mets_files(dataset, fold)
            if not mets_files:
                print(f"Skipping {dataset} / {fold} — no files found.")
                continue
            key = f"{dataset}__{fold}"
            all_metrics[key] = Metrics(
                dataset, fold,
                *get_global_metrics(mets_files, metric_files)
            )
            print(f"Loaded {key}")
    return all_metrics


In [ ]:
# ── cached region case lists ──────────────────────────────────────────────────

_REGION_CASE_CACHE = {}

def _get_region_cases(region: str) -> list:
    """
    Cache and return the list of case names that have data for a given region.
    These come from the unique_*_cases.csv files generated by the bone analysis pipeline.
    """
    global _REGION_CASE_CACHE
    if region in _REGION_CASE_CACHE:
        return _REGION_CASE_CACHE[region]

    region_csv_map = {
        'liver':    'unique_liver_cases.csv',
        'lung':     'unique_lung_cases.csv',
        'bones':    'unique_bone_cases.csv',
        'hr_bones': 'unique_hr_bone_cases.csv',
    }

    if region not in region_csv_map:
        raise ValueError(f"No case CSV defined for region: {region}")

    df = pd.read_csv(region_csv_map[region])
    df['file'] = df['file'].str.replace('.csv', '', regex=False)
    cases = df['file'].unique().tolist()
    _REGION_CASE_CACHE[region] = cases
    return cases


NO_LABEL_FILES = None

def _get_no_label_files() -> list:
    """Cache the no_label_files csv so we only read it once."""
    global NO_LABEL_FILES
    if NO_LABEL_FILES is None:
        nl = pd.read_csv("no_label_files.csv")
        nl['file_name'] = nl['file_name'].str.replace('.nii.gz', '', regex=False)
        NO_LABEL_FILES = nl['file_name'].tolist()
    return NO_LABEL_FILES


def _tracer_mask(df: pd.DataFrame, tracer: str) -> pd.Series:
    """Return a boolean mask filtering by tracer prefix in the 'file' column."""
    if tracer == 'ALL':
        return pd.Series([True] * len(df), index=df.index)
    return df['file'].str.lower().str.startswith(tracer.lower())


def _summarise_global_metrics(df: pd.DataFrame, key: str) -> dict:
    """
    Summarise global_metrics df (one row per case).
    Dice is calculated only on the already-filtered df, excluding no_label_files.
    Volumes are in cm3 (already converted upstream).
    df should already be filtered to the correct tracer + region cases before calling this.
    """
    empty = {
        'dataset__fold':      key,
        'n_cases':            0,
        'Dice':               float('nan'),
        'FP_vol_avg (cm3)':   float('nan'),
        'FN_vol_avg (cm3)':   float('nan'),
        'FP_vol_total (cm3)': float('nan'),
        'FN_vol_total (cm3)': float('nan'),
    }
    if df is None or len(df) == 0:
        return empty

    no_label  = _get_no_label_files()
    # Exclude no-label cases only for Dice — volumes use the full filtered set
    dice_df   = df[~df['file'].isin(no_label)]

    return {
        'dataset__fold':      key,
        'n_cases':            len(df),
        'Dice':               dice_df['dice_sc'].mean(),
        'FP_vol_avg (cm3)':   df['false_pos_vol'].mean(),
        'FN_vol_avg (cm3)':   df['false_neg_vol'].mean(),
        'FP_vol_total (cm3)': df['false_pos_vol'].sum(),
        'FN_vol_total (cm3)': df['false_neg_vol'].sum(),
    }


def _summarise_mets(df: pd.DataFrame, key: str) -> dict:
    """
    Summarise a mets df (liver / lung / bones / hr_bones).
    Aggregates per-case first so multi-lesion cases aren't over-weighted.
    Dice is calculated only on labelled cases, excluding no_label_files.
    df should already be filtered to the correct tracer + region cases before calling this.
    """
    empty = {
        'dataset__fold':      key,
        'n_cases':            0,
        'Dice':               float('nan'),
        'FP_vol_avg (cm3)':   float('nan'),
        'FN_vol_avg (cm3)':   float('nan'),
        'FP_vol_total (cm3)': float('nan'),
        'FN_vol_total (cm3)': float('nan'),
    }
    if df is None or len(df) == 0:
        return empty

    no_label = _get_no_label_files()

    # One row per case
    grp = df.groupby('file').agg(
        Dice   = ('Dice_Score',    'mean'),
        FP_vol = ('FP_Volume_mm3', 'mean'),
        FN_vol = ('FN_Volume_mm3', 'mean'),
    ).reset_index()

    dice_grp = grp[~grp['file'].isin(no_label)]

    return {
        'dataset__fold':      key,
        'n_cases':            len(grp),
        'Dice':               dice_grp['Dice'].mean(),
        'FP_vol_avg (cm3)':   dice_grp['FP_vol'].mean(),
        'FN_vol_avg (cm3)':   dice_grp['FN_vol'].mean(),
        'FP_vol_total (cm3)': dice_grp['FP_vol'].sum(),
        'FN_vol_total (cm3)': dice_grp['FN_vol'].sum(),
    }


def _build_region_table(all_metrics: dict, region: str, tracer: str) -> pd.DataFrame:
    """
    Build a summary DataFrame for one region + tracer combination.

    For 'overall': uses global_metrics directly (one row per case).
    For all other regions: 
        - pulls the case list from unique_*_cases.csv
        - filters global_metrics to only those cases  →  _summarise_global_metrics
          (same Dice calculation method as overall, scoped to region cases)
        - also filters the mets sub-df                →  _summarise_mets
          (for FP/FN volumes at the lesion level)

    region : 'overall' | 'liver' | 'lung' | 'bones' | 'hr_bones'
    tracer : 'ALL'     | 'fdg'   | 'psma'
    """
    rows = []

    for key, m in all_metrics.items():

        if region == 'overall':
            sub_global = m.global_metrics.copy()
            # Apply tracer filter
            sub_global = sub_global[_tracer_mask(sub_global, tracer)]
            row = _summarise_global_metrics(sub_global, key)

        else:
            # ── get the cases that have data for this region ──────────────────
            region_cases = _get_region_cases(region)

            # ── filter global_metrics to region cases + tracer ────────────────
            sub_global = m.global_metrics.copy()
            sub_global = sub_global[sub_global['file'].isin(region_cases)]
            sub_global = sub_global[_tracer_mask(sub_global, tracer)]

            # ── also filter the mets df (for lesion-level FP/FN volumes) ──────
            mets_map = {
                'liver':    m.global_liver,
                'lung':     m.global_lung,
                'bones':    m.global_bones,
                'hr_bones': m.global_hr_bones,
            }
            sub_mets = mets_map[region]

            if sub_mets is not None and len(sub_mets) > 0:
                sub_mets = sub_mets[sub_mets['file'].isin(region_cases)]
                sub_mets = sub_mets[_tracer_mask(sub_mets, tracer)]
            else:
                sub_mets = None

            # Dice comes from global_metrics scoped to region cases (same as overall)
            # FP/FN volumes come from the mets df (lesion-level granularity)
            g_row = _summarise_global_metrics(sub_global, key)
            m_row = _summarise_mets(sub_mets, key)

            row = {
                'dataset__fold':      key,
                'n_cases':            g_row['n_cases'],
                'Dice':               g_row['Dice'],               # from global_metrics
                'FP_vol_avg (cm3)':   m_row['FP_vol_avg (cm3)'],   # from mets
                'FN_vol_avg (cm3)':   m_row['FN_vol_avg (cm3)'],
                'FP_vol_total (cm3)': m_row['FP_vol_total (cm3)'],
                'FN_vol_total (cm3)': m_row['FN_vol_total (cm3)'],
            }

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).set_index('dataset__fold')


def _styled_table(df: pd.DataFrame, title: str):
    """Render a single colour-graded table with a title."""
    from IPython.display import display

    fmt       = {}
    dice_cols = []
    fp_cols   = []
    fn_cols   = []

    for col in df.columns:
        if col == 'Dice':
            fmt[col] = '{:.4f}'
            dice_cols.append(col)
        elif col.startswith('FP'):
            fmt[col] = '{:.2f}'
            fp_cols.append(col)
        elif col.startswith('FN'):
            fmt[col] = '{:.2f}'
            fn_cols.append(col)
        elif col == 'n_cases':
            fmt[col] = '{:,}'   

    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")

    styler = df.style.format(fmt, na_rep='—')

    if dice_cols:
        styler = styler.background_gradient(subset=dice_cols, cmap='RdYlGn',   axis=0)
    if fp_cols:
        styler = styler.background_gradient(subset=fp_cols,   cmap='RdYlGn_r', axis=0)
    if fn_cols:
        styler = styler.background_gradient(subset=fn_cols,   cmap='RdYlGn_r', axis=0)

    display(styler)


def display_all_summaries(all_metrics: dict):
    """
    Renders a table for every (tracer × region) combination, grouped by tracer.

    Layout:
        All Tracers → Overall | Liver | Lung | Bones | HR Bones
        PSMA        → Overall | Liver | Lung | Bones | HR Bones
        FDG         → Overall | Liver | Lung | Bones | HR Bones
    """
    TRACERS = [
        ('ALL',  'All Tracers'),
        ('psma', 'PSMA'),
        ('fdg',  'FDG'),
    ]
    REGIONS = [
        ('overall',  'Overall'),
        ('liver',    'Liver'),
        ('lung',     'Lung'),
        ('bones',    'Bones'),
        ('hr_bones', 'High Risk Bones'),
    ]

    for tracer_key, tracer_label in TRACERS:
        for region_key, region_label in REGIONS:
            df = _build_region_table(all_metrics, region=region_key, tracer=tracer_key)
            if df.empty:
                print(f"  [no data] {tracer_label} — {region_label}")
                continue
            _styled_table(df, title=f"{tracer_label}  |  {region_label}")

In [ ]:
# ── configure which datasets / folds you want ─────────────────────────────
DATASET_FOLDS = {
    "Dataset902_AutoPet": ["fold_0"],
    # add more as needed...
}

all_metrics = load_all_datasets(DATASET_FOLDS)
display_all_summaries(all_metrics)

Found 321 METS files in //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset902_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/bone_analysis/.
Loaded Dataset902_AutoPet__fold_0

  All Tracers  |  Overall


,n_cases,Dice,FP_vol_avg (cm3),FN_vol_avg (cm3),FP_vol_total (cm3),FN_vol_total (cm3)
dataset__fold,,,,,,
Dataset902_AutoPet__fold_0,321,0.5880,25.04,51.13,8037.72,16413.10



  All Tracers  |  Liver


c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax


,n_cases,Dice,FP_vol_avg (cm3),FN_vol_avg (cm3),FP_vol_total (cm3),FN_vol_total (cm3)
dataset__fold,,,,,,
Dataset902_AutoPet__fold_0,20,0.6172,—,—,—,—


...

c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax


,n_cases,Dice,FP_vol_avg (cm3),FN_vol_avg (cm3),FP_vol_total (cm3),FN_vol_total (cm3)
dataset__fold,,,,,,
Dataset902_AutoPet__fold_0,41,0.7136,—,—,—,—



  FDG  |  High Risk Bones


c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\BhattacharyaLab\anaconda3\envs\monai310\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax


,n_cases,Dice,FP_vol_avg (cm3),FN_vol_avg (cm3),FP_vol_total (cm3),FN_vol_total (cm3)
dataset__fold,,,,,,
Dataset902_AutoPet__fold_0,24,0.6733,—,—,—,—


In [ ]:

all_metrics